# How to calculate the Grundrentenzuschlag

The Grundrentenzuschlag (basic pension supplement) raises the statutory pensions of
people who paid contributions for at least 33 years at low earnings. It is reduced by
the pensioner's income (§ 97a SGB VI). This income is the sum of

- the zu versteuerndes Einkommen (taxable income),
- the steuerfreier Teil (tax-free part) of pensions, and
- capital income taxed under the Abgeltungsteuer,

taken as one twelfth per month.

The Grundrentenzuschlag does not use current income but the income of the calendar year
before the previous one (vorvergangenes Kalenderjahr, § 97a Abs. 2 S. 2 SGB VI). For the
Zuschlag paid in 2026, this is the income of 2024. This income can be calculated via a
separate GETTSIM run as illustrated in the example below.

### Overview

The calculation involves two steps:

1. Calculate the income relevant for the Grundrentenzuschlag in the year **before the
   previous** year. Use the target  
   `("sozialversicherung", "rente", "grundrente", "einkommen_für_einkommensanrechnung_im_übernächsten_jahr_m_ehe")`  
   with the policy environment of that year.

2. Use the result as the input  
   `("sozialversicherung", "rente", "grundrente", "einkommen_vorvorjahr_m_ehe")`  
   and calculate the Grundrentenzuschlag `("sozialversicherung", "rente", "grundrente", "betrag_m")`
   with the policy environment of the current year.

> **Note**: GETTSIM simplifies the statute in several ways:
>
> - If the tax office has no assessment for the year before the previous one, the
>   statute uses the income of the year before that (§ 97a Abs. 2 S. 3 SGB VI). It is up
>   to the user to pass the income of the right year.
> - If the tax office has no data on a person, the statute cuts gross pensions by fixed
>   percentages instead (§ 97a Abs. 2 S. 4 SGB VI). GETTSIM does not model this
>   fallback.
> - The tax-free part of Versorgungsbezüge (civil servant pensions) is not modelled
>   ([#1247](https://github.com/ttsim-dev/gettsim/issues/1247)).
> - The steuerfreier Teil of pensions is the fixed euro amount set when the pension
>   started (§ 22 Nr. 1 S. 3 Buchst. a Doppelbuchst. aa S. 4 EStG). GETTSIM approximates
>   it as one minus the Besteuerungsanteil times the current pension, which overstates
>   it for pensions that have been adjusted since they started.
> - For capital income under § 20 Abs. 1 Nr. 6 EStG, the statute counts one tenth of
>   the yield for at most ten years. GETTSIM counts it in full
>   ([#1248](https://github.com/ttsim-dev/gettsim/issues/1248)).

## Example

Consider a single pensioner, born in 1953, who has drawn her statutory pension since
2018. In 2024, she received a statutory pension of €1,250 and an occupational pension
(Betriebsrente) of €500 per month, plus €3,000 of capital income. She received no
Grundrentenzuschlag in 2024. We want her Grundrentenzuschlag in January 2026.

### Step 1: Income in the year before the previous year

In [ ]:
import numpy as np

from gettsim import InputData, MainTarget, TTTargets, main

First, we compile the data needed to calculate her income tax for 2024.

In [ ]:
input_data_2024 = {
    "alter": np.array([71]),
    "alter_monate": np.array([852]),
    "arbeitsstunden_w": np.array([0]),
    "behinderungsgrad": np.array([0]),
    "einkommensteuer": {
        "abzüge": {
            "beitrag_private_rentenversicherung_m": np.array([0]),
            "kinderbetreuungskosten_m": np.array([0]),
            "p_id_kinderbetreuungskostenträger": np.array([-1]),
        },
        "einkünfte": {
            "aus_forst_und_landwirtschaft": {"betrag_y": np.array([0])},
            "aus_gewerbebetrieb": {"betrag_y": np.array([0])},
            "aus_nichtselbstständiger_arbeit": {
                "tatsächliche_werbungskosten_y": np.array([0]),
            },
            "aus_selbstständiger_arbeit": {"betrag_y": np.array([0])},
            "aus_vermietung_und_verpachtung": {"betrag_y": np.array([0])},
            "ist_hauptberuflich_selbstständig": np.array([False]),
            "sonstige": {
                "alle_weiteren_y": np.array([0]),
                "rente": {
                    "alter_beginn_leistungsbezug_sonstige_private_vorsorge": np.array(
                        [65]
                    ),
                },
            },
        },
        "gemeinsam_veranlagt": np.array([False]),
    },
    "einnahmen": {
        "bruttolohn_m": np.array([0]),
        "kapitalerträge_y": np.array([3000]),
        "renten": {
            "aus_berufsständischen_versicherungen_m": np.array([0]),
            "basisrente_m": np.array([0]),
            "betriebliche_altersvorsorge_m": np.array([500]),
            "geförderte_private_vorsorge_m": np.array([0]),
            "gesetzliche_m": np.array([1250]),
            "sonstige_private_vorsorge_m": np.array([0]),
        },
    },
    "familie": {
        "alleinerziehend": np.array([False]),
        "p_id_ehepartner": np.array([-1]),
        "p_id_elternteil_1": np.array([-1]),
        "p_id_elternteil_2": np.array([-1]),
    },
    "geburtsjahr": np.array([1953]),
    "kindergeld": {
        "in_ausbildung": np.array([False]),
        "p_id_empfänger": np.array([-1]),
    },
    "p_id": np.array([0]),
    "sozialversicherung": {
        "kranken": {"beitrag": {"privat_versichert": np.array([False])}},
        "pflege": {"beitrag": {"hat_kinder": np.array([True])}},
        "rente": {
            "altersrente": {"betrag_m": np.array([1250.0])},
            "bezieht_altersrente": np.array([True]),
            "grundrente": {"betrag_m": np.array([0.0])},
            "jahr_renteneintritt": np.array([2018]),
            "verzichtet_auf_versicherungsfreiheit_wegen_alters": np.array([False]),
            "verzichtet_auf_versicherungsfreiheit_wegen_geringfügiger_beschäftigung": np.array(
                [False]
            ),
        },
    },
}

Now, we calculate the income relevant for the Grundrentenzuschlag in 2026, using the
policy environment of 2024.

In [ ]:
einkommen_2024 = main(
    main_target=MainTarget.results.tree,
    policy_date_str="2024-01-01",
    tt_targets=TTTargets.tree(
        {
            "sozialversicherung": {
                "rente": {
                    "grundrente": {
                        "einkommen_für_einkommensanrechnung_im_übernächsten_jahr_m_ehe": None,
                    }
                }
            }
        }
    ),
    input_data=InputData.tree(input_data_2024),
    include_warn_nodes=False,
)["sozialversicherung"]["rente"]["grundrente"][
    "einkommen_für_einkommensanrechnung_im_übernächsten_jahr_m_ehe"
]
einkommen_2024

### Step 2: Grundrentenzuschlag in the current year

Now we calculate the Grundrentenzuschlag for January 2026 with the policy environment
of that date. Her pension record has 35 years (420 months) of Grundrentenzeiten, which
are also her Grundrentenbewertungszeiten, with 17.5 Entgeltpunkte (0.5 per year) in
total. We pass a Zugangsfaktor of 1 as data, so we need not supply her full insurance
history.

In [ ]:
input_data_2026 = {
    "familie": {"p_id_ehepartner": np.array([-1])},
    "p_id": np.array([0]),
    "sozialversicherung": {
        "rente": {
            "altersrente": {"zugangsfaktor": np.array([1.0])},
            "grundrente": {
                "bewertungszeiten_monate": np.array([420]),
                "einkommen_vorvorjahr_m_ehe": einkommen_2024,
                "grundrentenzeiten_monate": np.array([420]),
                "mean_entgeltpunkte": np.array([17.5]),
            },
        },
    },
}

main(
    main_target=MainTarget.results.tree,
    policy_date_str="2026-01-01",
    tt_targets=TTTargets.tree(
        {
            "sozialversicherung": {
                "rente": {
                    "grundrente": {
                        "betrag_m": None,
                    }
                }
            }
        }
    ),
    input_data=InputData.tree(input_data_2026),
    include_warn_nodes=False,
)